In [31]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class SatResNet(nn.Module):
    def __init__(self, num_classes: int=10):
        """
        Initialize a Resnet18-based model for satellite image classification

        Parameters
        ----------
        num_classes : int, optional
            the number of target classes (defaults to 10)
        pretrained : bool, optional
            If true, uses the pretrained resnet18 model.
        """
        super().__init__()

        self.model = resnet18(weights=ResNet18_Weights.DEFAULT)
        in_features = self.model.fc.in_features
        # replace the final fully connected layer
        self.model.fc = nn.Linear(in_features=in_features, out_features=num_classes)

    def forward(self, x):
        return self.model(x)

In [33]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [34]:
model = SatResNet().to(device)
print(model)

SatResNet(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_run

In [35]:
from torchvision.datasets import EuroSAT
from torchvision.transforms import ToTensor
from torch.utils.data import random_split, DataLoader

data = EuroSAT(root='../../data/processed', transform=ToTensor(), download=False)

train_size = int(0.8 * len(data))
test_size = len(data) - train_size

train, test = random_split(data, [train_size, test_size])

In [36]:
batch_size = 32
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=False)

In [37]:
model = SatResNet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [40]:
n_epochs = 10

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # zero parameter gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward pass optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(train)
    print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {epoch_loss:.4f}')

    # save checkpoint
    checkpoint_path = f'../../models/satresnet_epoch_{epoch+1}.pth'
    torch.save(model.state_dict(), checkpoint_path)
    print(f'Model checkpoint saved to {checkpoint_path}')

Epoch [1/10], Loss: 0.0694
Model checkpoint saved to ../../models/satresnet_epoch_1.pth
Epoch [2/10], Loss: 0.0700
Model checkpoint saved to ../../models/satresnet_epoch_2.pth
Epoch [3/10], Loss: 0.0662
Model checkpoint saved to ../../models/satresnet_epoch_3.pth
Epoch [4/10], Loss: 0.0543
Model checkpoint saved to ../../models/satresnet_epoch_4.pth
Epoch [5/10], Loss: 0.0527
Model checkpoint saved to ../../models/satresnet_epoch_5.pth
Epoch [6/10], Loss: 0.0576
Model checkpoint saved to ../../models/satresnet_epoch_6.pth
Epoch [7/10], Loss: 0.0539
Model checkpoint saved to ../../models/satresnet_epoch_7.pth
Epoch [8/10], Loss: 0.0352
Model checkpoint saved to ../../models/satresnet_epoch_8.pth
Epoch [9/10], Loss: 0.0533
Model checkpoint saved to ../../models/satresnet_epoch_9.pth
Epoch [10/10], Loss: 0.0332
Model checkpoint saved to ../../models/satresnet_epoch_10.pth
